# 对分易自动签到 APK 构建
全部运行 → 授权 Drive → 等 20 分钟 → APK 存到网盘。关电脑不受影响。

In [ ]:
# 0. 挂载 Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 1. 安装系统依赖
!sudo apt-get update -qq
!sudo apt-get install -y -qq openjdk-17-jdk-headless autoconf automake libtool cmake zip unzip libffi-dev libssl-dev python3-pip git

In [ ]:
# 2. 安装 buildozer
!pip install -q buildozer cython

In [ ]:
# 3. 克隆项目
!rm -rf duifene-sign
!git clone https://github.com/Mafeixxn/duifene-sign.git
%cd duifene-sign

In [ ]:
# 4. 构建 APK（约 15-25 分钟，自动重试 3 次）
import subprocess, time
for i in [1, 2, 3]:
    print(f'>>> 第 {i} 次构建...')
    with open('build.log', 'w') as f:
        r = subprocess.run(['buildozer', 'android', 'debug'],
                           input=b'y\n', stdout=f, stderr=subprocess.STDOUT)
    if r.returncode == 0:
        print('成功！')
        break
    print(f'失败 (exit={r.returncode})')
    if i < 3:
        time.sleep(5)
    else:
        import shutil
        shutil.copy('build.log', '/content/drive/MyDrive/duifene_build_error.log')
        print('3次均失败，日志已存到 Drive')

In [ ]:
# 5. 显示错误（如有）
!grep -i -E 'error:|failed:|exception|traceback' build.log | tail -15 || echo '没有明显错误'

In [ ]:
# 6. 保存 APK 到 Google Drive
import glob, shutil, os
apks = glob.glob('bin/*.apk')
if apks:
    dest = '/content/drive/MyDrive/duifene_sign.apk'
    shutil.copy(apks[0], dest)
    mb = os.path.getsize(apks[0]) / (1024*1024)
    print(f'APK ({mb:.1f}MB) 已存到 Drive: duifene_sign.apk')
else:
    print('未找到 APK，构建可能失败')